In [0]:
import os
import sys
from pyspark.sql.functions import current_timestamp, input_file_name, col


parent_path = '/Workspace/Users/esymphony.life@gmail.com/MSF_test'
if parent_path not in sys.path:
    sys.path.insert(0, parent_path)

from customer.common.functions import ingest_to_delta, get_read_options

In [0]:

def get_uningested_files(volume_path, schema="bronze"):
    """
    Identifies all files in a Volume that have not yet been ingested into 
    the corresponding table (derived from the volume path).
    """
    # Parse volume_path to derive the catalog.schema.table
    _, _, catalog, domain, table_name = volume_path.split("/")
    target_table = f"{catalog}.{domain}.{schema}_{table_name}"   

    # Get current files in the volume
    current_files_df = (spark.read.format("binaryFile")
                        .option("recursiveFileLookup", "true")
                        .load(volume_path)
                        .select(col("path"), col("modificationTime")))

    # Check against the derived target table
    if spark.catalog.tableExists(target_table):
        # Select distinct source files already processed
        ingested_df = (spark.read.table(target_table)
                       .select(col("_source_file").alias("path"))
                       .distinct())
        
        # Filter out already processed files
        unprocessed_df = current_files_df.join(ingested_df, on="path", how="left_anti")
    else:
        # Initial ingestion
        unprocessed_df = current_files_df

    unprocessed_rows = (unprocessed_df
                        .orderBy(col("modificationTime").asc())
                        .select("path")
                        .collect())

    # Return a list of strings
    return [row.path[5:] for row in unprocessed_rows]


In [0]:

def ingest_to_delta(spark, filetype, file_path, table_path, read_options=None):
    """
    Ingest any file type to Delta table
    
    Args:
        filetype: File format (json, csv, parquet, avro, etc.)
        file_path: Source file path (can be volume path)
        table_path: Target Delta table name (catalog.schema.table)
        read_options: Optional dict of read options (e.g., {"multiLine": True, "header": True})
    """
    # Read with optional format-specific options
    reader = spark.read.format(filetype)
    if read_options:
        for key, value in read_options.items():
            reader = reader.option(key, value)
    
    df = reader.load(file_path)
    
    # Add metadata columns
    df = df.withColumn("_ingest_timestamp", lit(current_timestamp())) \
           .withColumn("_source_file", lit(file_path))
    
    # Write to Delta table
    df.write.format("delta").mode("append").saveAsTable(table_path)
    
    print(f"Successfully wrote to {table_path}")
    return df

In [0]:
def ingest_volume(vol_path):
    _, catalog, domain, volume = vol_path.split("/")
    schema = "bronze"

    write_path = f"{catalog}.{domain}.{schema}_{volume}"

    for file_path in get_uningested_files(volume_path):
        filetype = file_path.split(".")[-1]
        read_options = get_read_options(filetype)
        ingest_to_delta(spark, filetype, file_path, write_path, read_options)
    
    print(f"Successfully ingested all files from {volume}")

In [0]:
ingest_volume(dbutils.widgets.get("VOLUME_PATH"))